In [ ]:
# Get list of files
# Figure out columns I needfor river flooding for baseline
# Then have baseline damages
# Intersect with hydrobasins to give info - spatial join to get hydro basin IDs
# Then can say something about baseline risk in each hydrobasin

## Step 0: Import packages

In [ ]:
import os
from pathlib import Path

import pandas as pd
import geopandas as gpd

## Step 1: Set up base paths and other relevant paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
current_damages_path = base_path / "Processed_data/direct_damages_summary_uids"
hydrobasins_path = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
networks_path = base_path / "Processed_data/networks"

In [ ]:
# Define the directory where GeoPackage files were saved
output_directory = "catchment_outputs"

output_aggregated_path = Path("aggregated_hydrobasin_data")
output_aggregated_path.mkdir(exist_ok=True)  # Create the output directory

## Step 2: Load hydrobasins data

In [ ]:
# Load the hydrobasins data
hydrobasins = gpd.read_file(hydrobasins_path)
print(f"Loaded Hydrobasins: {len(hydrobasins)} polygons")

# Ensure hydrobasins has a consistent CRS
hydrobasins = hydrobasins.to_crs("EPSG:3448")  # CRS for Jamaica

## Step 3: Load the network data that we organised by catchments in the other notebook

In [ ]:
def load_catchment_data(output_dir):
    """
    Recursively load all GeoPackage files from the output directory.
    """
    catchment_gdfs = {}
    for root, _, files in os.walk(output_dir):
        for file in files:
            if file.endswith(".gpkg"):
                file_path = os.path.join(root, file)
                gdf = gpd.read_file(file_path)
                key = os.path.relpath(file_path, output_dir).replace(os.sep, "__")
                catchment_gdfs[key] = gdf
                print(f"Loaded {key}: {len(gdf)} features")
    return catchment_gdfs

##### Check how many network_catchment files we have so that I can later compare them with no. network_EAEL files to see if they align

In [ ]:
gpkg_files = []
for root, _, files in os.walk(output_directory):
    for file in files:
        if file.endswith(".gpkg"):
            gpkg_files.append(os.path.join(root, file))

print(f"Total GeoPackage files found: {len(gpkg_files)}")

##### Check the size of the larger 10 network_catchment files to see if the data quantities look about right

In [ ]:
file_sizes = []
for file in gpkg_files:
    size_mb = os.path.getsize(file) / (1024 * 1024)  # Convert bytes to MB
    file_sizes.append((file, size_mb))

# Sort by size (descending)
file_sizes.sort(key=lambda x: x[1], reverse=True)

# Display the largest files
for file, size in file_sizes[:10]:  # Show top 10 largest files
    print(f"{file}: {size:.2f} MB")

#### Load all network_catchment GeoPackage files

In [ ]:
catchment_gdfs = load_catchment_data(output_directory)
print(f"Loaded {len(catchment_gdfs)} catchment GeoDataFrames.")

In [ ]:
# # Function to load all GeoPackage files dynamically
# def load_catchment_data(output_dir):
#     catchment_gdfs = {}
#     for root, _, files in os.walk(output_dir):
#         for file in files:
#             if file.endswith(".gpkg"):
#                 # Load each GeoPackage file
#                 file_path = os.path.join(root, file)
#                 gdf = gpd.read_file(file_path)
#                 key = os.path.relpath(file_path, output_dir).replace(os.sep, "__")  # Create a unique key
#                 catchment_gdfs[key] = gdf
#                 print(f"Loaded {key}: {len(gdf)} features")
#     return catchment_gdfs


In [ ]:
# # Example: Access one GeoDataFrame
# sample_key = list(catchment_gdfs.keys())[0]
# sample_gdf = catchment_gdfs[sample_key]
# print(f"Columns in {sample_key}: {sample_gdf.columns}")
# print(sample_gdf.head())

## Step 4: Load the EAEL data 

#### Read in parquet files of the EAELs

In [ ]:
# List all .parquet files
parquet_files = sorted([f for f in os.listdir(current_damages_path) if f.endswith("EAEL.parquet")])
parquet_files

In [ ]:
# airport_fname = parquet_files[0]
# airport_df = pd.read_parquet(current_damages_path / airport_fname)
# airport_df.query('hazard == "fluvial" and rcp == "baseline"')

##### Check how many network_EAEL files we have so that I can compare them with no. network_catchment files to see if they align

In [ ]:
# Load EAEL parquet files
parquet_files = sorted([f for f in os.listdir(current_damages_path) if f.endswith("EAEL.parquet")])
print(f"Found {len(parquet_files)} EAEL Parquet files.")
for file in parquet_files:
    print(file)

##### Load all EAEL parquet files into DataFrames

In [ ]:
EAEL_fluvial_data = {}
for file in parquet_files:
    file_path = current_damages_path / file
    print(f"Processing EAEL file: {file}")
    
    # Load data
    eael_df = pd.read_parquet(file_path)

    # Apply filters
    fluvial_filtered = eael_df.query('hazard == "fluvial" and rcp == "baseline"')

    # Store in dictionary
    EAEL_fluvial_data[file] = fluvial_filtered
    print(f"  Filtered {len(fluvial_filtered)} rows for {file}")

In [ ]:
### This shows why we get 0 in electricity, cos it has no fluvial hazards


# # Load the electricity EAEL data
# elec_edges_file = current_damages_path / "electricity_network_v3.1_edges_EAD_EAEL.parquet"
# elec_edges_df = pd.read_parquet(elec_edges_file)

# # Print basic information
# print(f"Total Rows: {len(elec_edges_df)}")
# print("Columns:", elec_edges_df.columns)
# print(elec_edges_df.head())

# # Check unique hazard types
# print("\nUnique Hazard Types:", elec_edges_df["hazard"].unique())

# # Check unique RCP values
# print("\nUnique RCP Values:", elec_edges_df["rcp"].unique())

##### Have a look at the columns and data within the EAEL files

In [ ]:
for file in parquet_files:
    file_path = current_damages_path / file
    data = pd.read_parquet(file_path)
    print(f"Columns in {file}: {data.columns}")
    print(data.head(3))

In [ ]:
print("\n Available Catchment Layers:")
for key in catchment_gdfs.keys():
    print("-", key)

print("\n Expected Base Layer Names from EAEL Files:")
for file in EAEL_fluvial_data.keys():
    print("-", file.replace("_EAD_EAEL.parquet", ""))

In [ ]:
for file, df in EAEL_fluvial_data.items():
    print(f"{file}: {df.columns}")

In [ ]:
for file, gdf in catchment_gdfs.items():
    print(f"{file}: {gdf.columns}")

In [ ]:
# Define column name variations to be mapped to 'asset_id'
id_variations = ['node_id', 'edge_id', 'osm_id', 'id']

# Rename columns in EAEL_fluvial_data safely
for file, df in EAEL_fluvial_data.items():
    for col in id_variations:
        if col in df.columns:
            df = df.rename(columns={col: 'asset_id'})
    EAEL_fluvial_data[file] = df  # Assign back to the dictionary

# Verify the changes
for file, df in EAEL_fluvial_data.items():
    print(f"{file}: {df.columns}")

In [ ]:
# Identify any unusual prefixes in EAEL asset_id columns
for file, df in EAEL_fluvial_data.items():
    unique_prefixes = {str(x).split("_")[0] for x in df['asset_id'].dropna().unique() if isinstance(x, str) or "_" in str(x)}
    print(f"{file}: {unique_prefixes}")

In [ ]:
# Define standard prefix mappings for roads and rail
prefix_mapping = {
    "roads_nodes_EAD_EAEL.parquet": ("roadsn_", "roadn_"),  # Keep "roadn_" for nodes
    "roads_edges_EAD_EAEL.parquet": ("roadse_", "roade_"),  # Keep "roade_" for edges
    "rail_edges_EAD_EAEL.parquet": ("raile_", "raile_"),  # Ensure "raile_" is correctly formatted
    "rail_nodes_EAD_EAEL.parquet": ("railn_", "railn_"),  # Ensure "railn_" is correctly formatted
}

# Apply fixes for roads & rail
for file, (old_prefix, new_prefix) in prefix_mapping.items():
    if file in EAEL_fluvial_data:
        df = EAEL_fluvial_data[file]

        print(f"\nBefore Fix - {file} Sample IDs:")
        print(df['asset_id'].dropna().unique()[:10])  # Print first 10 IDs before change

        # Replace incorrect prefixes
        df['asset_id'] = df['asset_id'].str.replace(f"^{old_prefix}", new_prefix, regex=True)

        print(f"After Fix - {file} Sample IDs:")
        print(df['asset_id'].dropna().unique()[:10])  # Print first 10 IDs after change

        # Store back the modified DataFrame
        EAEL_fluvial_data[file] = df

# Standardize naming for pipelines, potable water, and wastewater facilities
for file in ["pipelines_NWC_edges_EAD_EAEL.parquet", "potable_facilities_NWC_nodes_EAD_EAEL.parquet", "waste_water_facilities_NWC_nodes_EAD_EAEL.parquet"]:
    if file in EAEL_fluvial_data:
        df = EAEL_fluvial_data[file]

        # Ensure consistent formatting before renaming
        df['asset_id'] = df['asset_id'].astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)

        print(f"\nBefore Fix - {file} Sample IDs:")
        print(df['asset_id'].dropna().unique()[:10])  # Print first 10 IDs before change

        # Update **pipeline** prefixes
        if file == "pipelines_NWC_edges_EAD_EAEL.parquet":
            df['asset_id'] = df['asset_id'].str.replace("^potable_", "pipe_potable_", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^sewer_", "pipe_sewer_", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^gravity_", "pipe_gravity_", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^pressure_", "pipe_pressure_", regex=True)

        # Update **potable water** facility prefixes
        if file == "potable_facilities_NWC_nodes_EAD_EAEL.parquet":
            potable_prefixes = {
                "pump_": "potable_water_pump_",
                "booster_": "potable_water_booster_",
                "treatment_": "potable_water_treatment_",
                "filter_": "potable_water_filter_",
                "river_": "potable_water_river_",
                "spring_": "potable_water_spring_",
                "relift_": "potable_water_relift_",
                "production_": "potable_water_production_",
                "sump_": "potable_water_sump_"
            }
            for old_prefix, new_prefix in potable_prefixes.items():
                df['asset_id'] = df['asset_id'].str.replace(f"^{old_prefix}", new_prefix, regex=True)

        # Update **wastewater facility** prefixes (AFTER lowercase conversion)
        if file == "waste_water_facilities_NWC_nodes_EAD_EAEL.parquet":
            df['asset_id'] = df['asset_id'].str.replace("^ww_pump_station", "WW_pump_station", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^ww_relift_station", "WW_relift_station", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^ww_treatment_plant", "WW_treatment_plant", regex=True)
            df['asset_id'] = df['asset_id'].str.replace("^sump_", "WW_sump_", regex=True)  # Fix Sump issue

        print(f"After Fix - {file} Sample IDs:")
        print(df['asset_id'].dropna().unique()[:10])  # Print first 10 IDs after change

        # Store back the modified DataFrame
        EAEL_fluvial_data[file] = df

In [ ]:
# Dictionary to store merged results
merged_data = {}

# Ensure asset_id is always a clean string in both datasets
for file, df in EAEL_fluvial_data.items():
    df = df.copy()  # Avoid modifying a view

    # Convert numeric IDs to string explicitly and remove trailing `.0`
    df['asset_id'] = df['asset_id'].astype(str).str.strip().str.lower()
    df['asset_id'] = df['asset_id'].str.replace(r'\.0$', '', regex=True)

    # Fix **potable water facilities** missing `potable_water_` prefix
    if "potable_facilities_NWC_nodes_EAD_EAEL.parquet" in file:
        df['asset_id'] = df['asset_id'].str.replace("^pump_", "potable_water_pump_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^booster_", "potable_water_booster_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^treatment_", "potable_water_treatment_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^filter_", "potable_water_filter_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^river_", "potable_water_river_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^spring_", "potable_water_spring_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^relift_", "potable_water_relift_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^production_", "potable_water_production_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^sump_", "potable_water_sump_", regex=True)

    # Fix **pipelines**: Ensure all potable pipelines have the correct prefix
    if "pipelines_NWC_edges_EAD_EAEL.parquet" in file:
        df['asset_id'] = df['asset_id'].str.replace("^potable_", "pipe_potable_", regex=True)
        df['asset_id'] = df['asset_id'].str.replace("^sewer_", "pipe_sewer_", regex=True)

    # Fix **electricity network**: Ensure `node_` and `edge_` prefixes match
    if "electricity_network_v3.1_nodes_EAD_EAEL.parquet" in file:
        df['asset_id'] = df['asset_id'].str.replace("^node_", "node_", regex=True)  # Enforce consistency
    if "electricity_network_v3.1_edges_EAD_EAEL.parquet" in file:
        df['asset_id'] = df['asset_id'].str.replace("^edge_", "edge_", regex=True)  # Enforce consistency

    EAEL_fluvial_data[file] = df  # Store back in dictionary

for file, gdf in catchment_gdfs.items():
    gdf = gdf.copy()  # Avoid modifying a view

    # Convert numeric asset_id values to string and remove `.0`
    if gdf['asset_id'].dtype in ['int64', 'float64']:
        gdf['asset_id'] = gdf['asset_id'].astype(int).astype(str)

    gdf['asset_id'] = gdf['asset_id'].astype(str).str.strip().str.lower()
    gdf['asset_id'] = gdf['asset_id'].str.replace(r'\.0$', '', regex=True)

    catchment_gdfs[file] = gdf  # Store back in dictionary

# Perform the merge
for eael_file, df in EAEL_fluvial_data.items():
    base_name = eael_file.replace("_EAD_EAEL.parquet", "")
    matching_gdf_file = next((key for key in catchment_gdfs.keys() if base_name in key), None)

    if matching_gdf_file:
        print(f"\nMerging {eael_file} with {matching_gdf_file}")

        # Merge the EAEL data with the corresponding catchment GeoDataFrame
        merged_df = df.merge(catchment_gdfs[matching_gdf_file], on="asset_id", how="inner")

        # Store the merged DataFrame
        merged_data[eael_file] = merged_df

        print(f"  Merged {len(merged_df)} rows")

# Verify merged results
for file, df in merged_data.items():
    print(f"{file}: {len(df)} rows after merge")

# Check missing asset IDs
for eael_file, df in EAEL_fluvial_data.items():
    base_name = eael_file.replace("_EAD_EAEL.parquet", "")
    matching_gdf_file = next((key for key in catchment_gdfs.keys() if base_name in key), None)

    if matching_gdf_file:
        eael_ids = set(df['asset_id'])
        gdf_ids = set(catchment_gdfs[matching_gdf_file]['asset_id'])

        missing_in_gdf = eael_ids - gdf_ids  # IDs in EAEL but NOT in catchment data
        missing_in_eael = gdf_ids - eael_ids  # IDs in catchment data but NOT in EAEL

        print(f"\n{eael_file} - {matching_gdf_file}:")
        print(f"  {len(missing_in_gdf)} asset IDs in EAEL but not in catchment data")
        print(f"  {len(missing_in_eael)} asset IDs in catchment data but not in EAEL")

        # Print a few sample IDs
        if missing_in_gdf:
            print(f"  Sample missing in catchment: {list(missing_in_gdf)[:10]}")
        if missing_in_eael:
            print(f"  Sample missing in EAEL: {list(missing_in_eael)[:10]}")